# S-VLG — Huấn luyện V2 (SU-MedVQA) THẬT trên Colab GPU

Notebook này train model **thật** — Qwen2.5-3B-Instruct + QLoRA 4-bit — trên toàn bộ
VQA-RAD + SLAKE, khác với `scripts/run_smoketest_v2.py` chạy local trước đó (model tí hon
`tiny-gpt2` trên CPU, chỉ để kiểm chứng pipeline). Đây chính là bước tạo số liệu **final**
để điền vào bài báo khoa học.

## Bước 0 — Chuẩn bị trên Google Drive (làm 1 lần, TRƯỚC khi mở notebook này)

1. Upload toàn bộ thư mục project `S-VLG` lên Google Drive của bạn (ví dụ `MyDrive/S-VLG`),
   gồm: `src/`, `scripts/`, `configs/`, `requirements.txt`, và **quan trọng nhất**:
   `data/raw/vqa-rad/` + `data/raw/slake/` (đã tải sẵn ở máy local, ~280MB tổng).
2. KHÔNG cần upload `.git/`, `__pycache__/`, `outputs/checkpoints/` (notebook tự tạo `outputs/`).
3. Ở Colab: Runtime → Change runtime type → Hardware accelerator → GPU → chọn **A100**.

Sửa biến `DRIVE_PROJECT_DIR` ở cell dưới nếu bạn đặt project ở vị trí khác `MyDrive/S-VLG`.

## Quy trình 3 bước
1. **Calibration**: chạy 1 epoch thật trên subset nhỏ để đo tốc độ THẬT trên A100 bạn đang dùng
   (số đo trên CPU/tiny-model KHÔNG dùng để suy ra thời gian GPU/Qwen được — kiến trúc decoder
   khác nhau hoàn toàn về quy mô).
2. **Full run**: toàn bộ VQA-RAD+SLAKE, N epoch, đủ 4 biến thể ablation.
3. **Compile**: gộp mọi số liệu vào `outputs/PAPER_DATA_v2.md` — file dùng để điền bài báo.

Checkpoint được lưu SAU MỖI EPOCH — nếu Colab ngắt kết nối giữa chừng, xem mục **Resume** ở cuối
notebook thay vì chạy lại từ đầu.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("CUDA available:", torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Sửa đường dẫn này nếu bạn đặt project ở vị trí khác trên Drive.
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/S-VLG"
LOCAL_PROJECT_DIR = "/content/S-VLG"

import os
assert os.path.isdir(DRIVE_PROJECT_DIR), f"Không thấy {DRIVE_PROJECT_DIR} — kiểm tra lại Bước 0 (đã upload project lên Drive đúng chỗ chưa?)"

# Copy code + data TỪ Drive VỀ ổ đĩa local của Colab: đọc/ghi nhanh hơn Drive FUSE mount
# nhiều lần, quan trọng vì SLAKE có hơn 600 thư mục ảnh đọc lặp lại mỗi epoch.
# KHÔNG exclude outputs/checkpoints — nếu bạn đang RESUME sau khi bị ngắt, checkpoint cũ
# đã được đồng bộ lên Drive ở lần chạy trước cần được copy về đây.
os.makedirs(LOCAL_PROJECT_DIR, exist_ok=True)
!rsync -a --exclude='.git' --exclude='__pycache__' "$DRIVE_PROJECT_DIR"/ "$LOCAL_PROJECT_DIR"/

%cd $LOCAL_PROJECT_DIR
!ls

## Bước 0.5 — Đồng bộ CODE mới nhất từ GitHub (đè lên bản rsync từ Drive ở trên)

`data/raw/` (VQA-RAD + SLAKE) vẫn phải lấy từ Drive như Bước 0 — dữ liệu bị `.gitignore`,
không nằm trên GitHub. Nhưng **code** (`src/`, `scripts/`, `configs/`, `requirements.txt`)
giờ lấy thẳng từ GitHub (`https://github.com/minhhaidhsp/s-vlg`) để mỗi lần có bugfix bên máy
local, bạn chỉ cần chạy lại cell dưới đây — KHÔNG cần re-upload lại toàn bộ project lên Drive.

**Chỉ `checkout` đúng 4 thư mục/file code** (`src scripts configs requirements.txt`), KHÔNG
`reset --hard` toàn bộ cây — vì `outputs/tables/`, `outputs/figures/`, `outputs/PAPER_DATA_v2.md`
CÓ được track trong git (chứa số liệu pilot cũ từ máy local), `reset --hard` sẽ xóa mất kết quả
train thật của bạn nếu chạy lại cell này giữa/sau một lần train dài trên Colab.

In [ ]:
REPO_URL = "https://github.com/minhhaidhsp/s-vlg.git"

%cd $LOCAL_PROJECT_DIR
if not os.path.isdir(".git"):
    !git init -q
    !git remote add origin {REPO_URL}
!git fetch origin master -q

# Chỉ ghi đè CODE (src/scripts/configs/requirements.txt) từ GitHub — không đụng outputs/,
# data/, notes.txt hay bất kỳ file nào khác đang có trên đĩa Colab.
!git checkout origin/master -- src scripts configs requirements.txt
print("Code đã đồng bộ tới commit:")
!git log origin/master -1 --oneline

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# Sanity check: dữ liệu và GPU đều sẵn sàng trước khi tốn thời gian train.
import json
from pathlib import Path

for name in ["vqa-rad", "slake"]:
    meta_path = Path(f"data/raw/{name}/metadata.json")
    assert meta_path.exists(), f"Thiếu {meta_path} — kiểm tra lại Bước 0 (đã upload data/raw/ lên Drive chưa?)"
    data = json.loads(meta_path.read_text(encoding="utf-8"))
    print(f"{name}: {len(data)} cặp QA")

import torch
assert torch.cuda.is_available(), "KHÔNG có GPU — kiểm tra lại Runtime type (phải chọn GPU/A100)!"
print("GPU:", torch.cuda.get_device_name(0))

## Bước 1 — Calibration: đo thời gian thật cho 1 epoch

Chạy 1 epoch thật (model Qwen2.5+QLoRA thật, `--real`) trên subset nhỏ (200 mẫu/bộ, `--n 200`)
để đo tốc độ thật trên chính GPU bạn đang dùng. `--skip-ablation` bỏ qua 4 lần train ablation
(không cần cho calibration, tiết kiệm thời gian). Có progress bar theo batch.

In [ ]:
!python scripts/run_smoketest_v2.py --real --epochs 1 --n 200 --skip-ablation

## Bước 2 — Ước lượng thời gian full run

Đọc dòng `Trained 1 epoch(s) this run in Xs (Xs/epoch)` ở output cell trên.

Full dataset (VQA-RAD 2.244 + SLAKE 7.033 = 9.277 cặp QA) lớn hơn subset calibration
(1.000 cặp QA, `--n 200`×2) khoảng **9 lần**. Ước lượng thô cho N epoch:

```
thời gian ước tính ≈ (Xs/epoch đo được ở Bước 1) × 9 × N
```

Đây CHỈ là suy diễn tuyến tính đơn giản — thực tế A100 có thể tận dụng batch tốt hơn ở scale
lớn hơn (nhanh hơn ước lượng) hoặc bị giới hạn bởi I/O đọc ảnh (chậm hơn). Coi là tham khảo,
không phải cam kết. Cộng thêm ~4× thời gian 1 epoch nữa cho 4 biến thể ablation (mỗi biến thể
mặc định 1 epoch, xem `--ablation-epochs`).

In [ ]:
def sync_outputs_to_drive():
    """Đồng bộ outputs/ (checkpoints, tables, PAPER_DATA_v2.md) về Google Drive.
    Gọi hàm này bất cứ lúc nào để lưu tạm — quan trọng nhất là sau mỗi lần train dài,
    để không mất kết quả nếu Colab ngắt kết nối ngay sau đó."""
    !rsync -a "$LOCAL_PROJECT_DIR"/outputs/ "$DRIVE_PROJECT_DIR"/outputs/
    print("Đã đồng bộ outputs/ về Google Drive.")

sync_outputs_to_drive()

## Bước 3 — Chạy FULL: toàn bộ dataset, N epoch, đủ 4 biến thể ablation

- `--n 0`: dùng TOÀN BỘ VQA-RAD + SLAKE (không lấy subset).
- `EPOCHS`: sửa theo ước lượng ở Bước 2 và budget thời gian bạn có (đề xuất 10).
- Checkpoint lưu SAU MỖI EPOCH vào `outputs/checkpoints/v2_real/` — nếu bị ngắt giữa chừng,
  dùng mục **Resume** ở cuối notebook thay vì chạy lại từ đầu.
- **Mỗi lần chạy script sẽ THÊM bản ghi mới vào `outputs/tables/v2/*.json`, không ghi đè** — nếu
  bạn đã chạy calibration hoặc chạy thử ở trên với `--n 200`, các bảng số liệu sẽ có thêm dòng
  đó. `compile_paper_data.py` (Bước 4) tự lấy bản ghi MỚI NHẤT cho mỗi model/dataset khi không
  đủ ≥2 seed để tính mean±std, nên số liệu cuối cùng vẫn đúng — nhưng nếu muốn bảng sạch hoàn
  toàn chỉ có 1 lần chạy thật, hãy xoá `outputs/tables/v2/` và `outputs/figures/data/v2/` trên
  Drive trước khi chạy Bước 3.

⚠️ Đây là lần chạy dài (có thể vài chục phút đến vài giờ tuỳ EPOCHS) — giữ tab Colab mở.

In [ ]:
EPOCHS = 10  # sửa nếu muốn khác, dựa trên ước lượng ở Bước 2

!python scripts/run_smoketest_v2.py --real --epochs {EPOCHS} --n 0

sync_outputs_to_drive()

## Bước 4 — Xem kết quả

`compile_paper_data.py` đã tự chạy ở cuối Bước 3 (là bước cuối của `run_smoketest_v2.py`).
In lại nội dung `outputs/PAPER_DATA_v2.md` ở đây cho tiện đọc/copy.

In [ ]:
print(Path("outputs/PAPER_DATA_v2.md").read_text(encoding="utf-8"))

## Resume — nếu Colab ngắt kết nối giữa chừng Bước 3

1. Chạy lại từ đầu notebook: mount Drive → copy project về local (cell ở Bước 0/setup) — vì
   không exclude `outputs/checkpoints` nên checkpoint đã đồng bộ lên Drive lần trước (qua
   `sync_outputs_to_drive()`) sẽ được copy về local.
2. Chạy 2 cell dưới đây thay cho Bước 3.

Lưu ý: `--resume-from` chỉ khôi phục việc train model chính (Bước b); nếu bị ngắt trong lúc
chạy ablation (Bước d), script sẽ chạy lại cả 4 biến thể ablation từ đầu khi rerun (mỗi biến
thể chỉ 1 epoch nên chi phí chạy lại không lớn).

In [ ]:
import glob

ckpts = sorted(glob.glob(f"{LOCAL_PROJECT_DIR}/outputs/checkpoints/v2_real/full/*.pt"))
print("Checkpoint tìm thấy:", ckpts)
RESUME_FROM = ckpts[-1] if ckpts else None
print("Sẽ resume từ:", RESUME_FROM)
assert RESUME_FROM is not None, "Không tìm thấy checkpoint nào — có thể Bước 3 chưa lưu được epoch nào, chạy lại Bước 3 bình thường."

In [ ]:
# EPOCHS phải GIỮ NGUYÊN mục tiêu tổng như lần chạy trước (vd 10) — script tự tính số epoch
# còn thiếu (nếu checkpoint đã ở epoch 6/10 thì chỉ train tiếp 7..10, không train lại từ đầu).
!python scripts/run_smoketest_v2.py --real --epochs {EPOCHS} --n 0 --resume-from "{RESUME_FROM}"

sync_outputs_to_drive()

## Sau khi hài lòng với số liệu: đánh dấu FINAL

Mọi số liệu ghi ra đều có `status="provisional"`. Khi đã chạy đủ epoch/seed bạn muốn dùng làm
số chính thức cho bài báo, chạy đoạn dưới (đổi tham số cho đúng bảng/model/seed) để nâng cấp
lên `"final"` — khi đó `compile_paper_data.py` sẽ in nhãn `[FINAL]` thay vì `[TẠM epoch=N]`.

In [ ]:
from src.utils.results_logger import ResultsLogger

logger = ResultsLogger(experiment_version="v2")
logger.mark_final("table6_overall", key_value="SU-MedVQA", key_field="model_name", dataset="vqa-rad+slake")
# Lặp lại cho các bảng khác nếu cần: table7_by_category, table9_ablation (key_field="variant_name"),
# table10_risk_coverage (key_field="config_name"), table11_efficiency.

!python scripts/compile_paper_data.py --version v2
sync_outputs_to_drive()